# Actividad 3 · Ensamblando la molécula
### De Lennard-Jones a un campo de fuerza completo (estilo MMFF94)
**Taller:** Física y matemática detrás de los campos de fuerza y la dinámica molecular · XXXIV Semana de la Química, Bogotá 2026

---

**Duración:** ~25 min · **Herramienta:** Google Colab o Jupyter · **Prerrequisito:** Actividades 1 y 2

**Objetivos:**
1. Ver que un campo de fuerza completo no es más que **sumar términos sencillos**.
2. Construir la energía de una molécula triatómica: enlaces + ángulo + Lennard-Jones.
3. Descubrir que los términos **compiten** y que el mínimo real es un compromiso.
4. Entender qué distingue a MMFF94: **los parámetros**, calibrados con experimentos.

## Concepto clave

Un campo de fuerza completo se ve así:

$$U = \underbrace{\sum k_b(r-r_0)^2}_{\text{enlaces}} + \underbrace{\sum k_\theta(\theta-\theta_0)^2}_{\text{\'angulos}} + \underbrace{\sum V_n[1+\cos(n\phi-\gamma)]}_{\text{diedros}} + \underbrace{\sum 4\varepsilon\left[\left(\tfrac{\sigma}{r}\right)^{12}-\left(\tfrac{\sigma}{r}\right)^{6}\right]}_{\text{vdW}} + \underbrace{\sum \frac{q_i q_j}{4\pi\varepsilon_0 r_{ij}}}_{\text{electrost\'atica}}$$

- Los **enlaces y ángulos son resortes**: la misma matemática de la Actividad 1, solo cambia la variable ($r \rightarrow \theta$).
- El término de **van der Waals es Lennard-Jones**: ya lo programaste.
- **¿Qué distingue a MMFF94?** No la forma de la función, sino los **parámetros**: $k$, $r_0$, $\varepsilon$, $\sigma$ y cargas que provienen de ajustes a estructuras y propiedades **experimentales** de moléculas orgánicas pequeñas. Por eso es el estándar para preparar ligandos.

> 🔑 **Un campo de fuerza = funciones físicas + parámetros calibrados para un tipo de molécula.**

## Paso 0 · Preparativos

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

## Paso 1 · El enlace químico es un resorte — *COMPLETA AQUÍ*

En los campos de fuerza, un enlace químico se modela como un **resorte**:

$$U_{\text{enlace}}(r) = k\,(r - r_0)^2$$

- $r_0$: longitud de equilibrio del enlace (donde $U = 0$, el resorte "descansado").
- $k$: rigidez del enlace. Enlaces reales tienen $k$ grande ($\sim$ 300–700 kcal/mol/Å²): **difíciles de estirar o comprimir**.

> 🤔 Derivación rápida: ¿cuál es la fuerza del enlace, $F = -dU/dr$? Deberías obtener $F = -2k(r-r_0)$: proporcional al estiramiento y con signo restaurador, como todo buen resorte.

Implementa el potencial. En Python, elevar al cuadrado es `x**2`.

In [ ]:
def U_enlace(r, k, r0):
    """Energía de un enlace: resorte armónico U = k*(r - r0)^2."""
    # COMPLETA AQUÍ: return ...
    return k * (r - r0)**2

# Prueba: un enlace con k = 500 kcal/mol/A^2 y r0 = 1.1 A
print("U(1.1) =", U_enlace(1.1, 500.0, 1.1), " <- en el equilibrio (debe ser 0)")
print("U(1.2) =", U_enlace(1.2, 500.0, 1.1), " <- estirado 0.1 A")
print("U(1.0) =", U_enlace(1.0, 500.0, 1.1), " <- comprimido 0.1 A (misma energía: simetría del resorte)")

## Paso 2 · Los otros términos y la molécula completa — *COMPLETA AQUÍ*

Construyamos una molécula triatómica **A–B–C** (como una versión miniatura de un ligando). Su energía total suma cuatro contribuciones:

1. Enlace A–B (resorte)
2. Enlace B–C (resorte)
3. Ángulo A–B–C (resorte angular: $k_\alpha(\theta-\theta_0)^2$)
4. Lennard-Jones entre **A y C**, que no están enlazados (los átomos enlazados no interactúan por vdW: es una convención de los campos de fuerza)

La geometría: B en el origen, A sobre el eje $x$ a distancia $r_1$, C a distancia $r_2$ formando ángulo $\theta$ en B. Por trigonometría:
$$AC = \sqrt{r_1^2 + r_2^2 - 2\,r_1 r_2 \cos\theta}$$

Las funciones de ángulo, Lennard-Jones y distancia ya están programadas. Tu tarea: **escribir la suma** en `U_molecula`.

In [ ]:
def U_angulo(theta, ka, th0):
    """Resorte angular: ka*(theta - th0)^2 con ángulos en grados."""
    return ka * (np.radians(theta) - np.radians(th0))**2

def U_LJ(r, eps, sig):
    """Lennard-Jones (Actividad 1)."""
    return 4 * eps * ((sig / r)**12 - (sig / r)**6)

def distancia_AC(r1, r2, theta):
    """Distancia A...C por ley de cosenos (theta en grados)."""
    th = np.radians(theta)
    return np.sqrt(r1**2 + r2**2 - 2*r1*r2*np.cos(th))

# ---- Parámetros de nuestra "molécula" (valores estilo real, kcal/mol, A, grados) ----
k_en, r0_en = 500.0, 1.1     # enlaces A-B y B-C: rígidos, 1.1 A
k_ang, th0  = 80.0, 120.0    # ángulo A-B-C
eps, sig    = 0.3, 2.2       # Lennard-Jones entre A y C

def U_molecula(r1, r2, theta):
    """Energía total de la molécula A-B-C: SUMA de los cuatro términos."""
    # COMPLETA AQUÍ: return U_enlace(...) + U_enlace(...) + U_angulo(...) + U_LJ(...)
    return (U_enlace(r1, k_en, r0_en) + U_enlace(r2, k_en, r0_en)
            + U_angulo(theta, k_ang, th0) + U_LJ(distancia_AC(r1, r2, theta), eps, sig))

print("U(1.1, 1.1, 120) =", round(U_molecula(1.1, 1.1, 120.0), 4), "kcal/mol  <- geometría de referencia")

## Experimento 1 · ¿Dónde está el mínimo? Los términos compiten

Vamos a explorar el paisaje de la molécula. Fijamos $r_1 = r_2 = 1.1$ Å y variamos el ángulo; luego fijamos $\theta = 120°$ y variamos $r_1$.

**Predicción previa:** ¿el mínimo estará exactamente en $\theta = 120°$ y $r = 1.1$ Å (los valores "desnudos" de los resortes), o se correrá un poco? Recuerda que el término de Lennard-Jones entre A y C **prefiere que se alejen** (menos solapamiento = menos repulsión), mientras que los resortes **prefieren sus valores de equilibrio**.

In [ ]:
# --- Paisaje angular ---
thetas = np.linspace(60, 180, 400)
U_th = U_molecula(r0_en, r0_en, thetas)
i_th = np.argmin(U_th)

# --- Paisaje del enlace ---
r1s = np.linspace(0.8, 1.8, 400)
U_r1 = U_molecula(r1s, r0_en, th0)
i_r = np.argmin(U_r1)

print(f"Mínimo angular:  theta = {thetas[i_th]:.1f}°   (resorte solo pediría {th0}°)")
print(f"Mínimo de enlace: r1  = {r1s[i_r]:.3f} Å  (resorte solo pediría {r0_en} Å)")

fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))
axes[0].plot(thetas, U_th, color='darkred', lw=2)
axes[0].axvline(th0, color='gray', ls='--', lw=1, label=f'resorte solo: {th0}°')
axes[0].axvline(thetas[i_th], color='green', ls=':', lw=1.5, label=f'mínimo real: {thetas[i_th]:.1f}°')
axes[0].set_xlabel('ángulo θ [°]'); axes[0].set_ylabel('U [kcal/mol]')
axes[0].set_title('Los términos compiten: el ángulo real no es 120°')
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(r1s, U_r1, color='navy', lw=2)
axes[1].axvline(r0_en, color='gray', ls='--', lw=1, label=f'resorte solo: {r0_en} Å')
axes[1].axvline(r1s[i_r], color='green', ls=':', lw=1.5, label=f'mínimo real: {r1s[i_r]:.3f} Å')
axes[1].set_xlabel('distancia r1 [Å]'); axes[1].set_ylabel('U [kcal/mol]')
axes[1].set_title('Idem para la longitud de enlace')
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout(); plt.show()

print("\n💡 Lectura: el mínimo real difiere de los valores de los resortes porque el Lennard-Jones A...C empuja a A y C a separarse.")

## Experimento 2 · Relajar la molécula: esto es lo que hace MMFF94

Una molécula "cruda" (por ejemplo, convertida de un SMILES) suele tener geometrías imperfectas. El paso **"minimizar MMFF94"** del flujo de docking es exactamente esto: dejar que el campo de fuerza la relaje hasta el mínimo más cercano.

Usamos el descenso de gradiente de la Actividad 2, pero con una variante práctica: en lugar de derivar a mano la energía total (¡sería tedioso!), la computadora calcula la derivada **numéricamente** (diferencias finitas):

$$F(r_1) \approx -\frac{U(r_1 + h) - U(r_1 - h)}{2h}$$

Partimos del enlace A–B estirado a 1.5 Å y lo dejamos relajar. *Completa el paso de actualización* (es el mismo de la Actividad 2).

In [ ]:
def fuerza_numerica(f, x, h=1e-4):
    """Fuerza = -dU/dx por diferencias finitas (la computadora deriva por nosotros)."""
    return -(f(x + h) - f(x - h)) / (2 * h)

# Energía vista como función solo de r1 (con r2 y theta fijos en sus valores de referencia)
def U_solo_r1(r1):
    return U_molecula(r1, r0_en, th0)

r = 1.5            # enlace estirado (geométricamente "mal")
alpha = 0.0005     # paso pequeño: las energías moleculares cambian rápido
trayectoria = [r]
for paso in range(20000):
    F = fuerza_numerica(U_solo_r1, r)
    # COMPLETA AQUÍ: actualiza r (descenso de gradiente, igual que en la Actividad 2)
    r = r + alpha * F
    trayectoria.append(r)
    if abs(F) < 1e-10:
        break

print(f"Relajación convergió en {len(trayectoria)-1} pasos")
print(f"r1 final = {r:.4f} Å   (coincide con el mínimo del Experimento 1: {r1s[i_r]:.3f} Å)")
print(f"Energía final = {U_solo_r1(r):.4f} kcal/mol")

fig, ax = plt.subplots(figsize=(8, 4.2))
ax.plot(r1s, U_r1, color='gray', lw=1.5, label='paisaje U(r1)')
ax.plot(trayectoria, U_solo_r1(np.array(trayectoria)), 'o-', color='darkred',
        ms=4, lw=1, label='relajación (descenso de gradiente)')
ax.annotate('geométria inicial\n(enlace estirado)', xy=(1.5, U_solo_r1(1.5)), xytext=(1.55, 12),
            arrowprops=dict(arrowstyle='->', color='black'))
ax.set_xlabel('r1 [Å]'); ax.set_ylabel('U [kcal/mol]')
ax.set_title('Así se "minimiza con MMFF94" un ligando antes del docking')
ax.legend(); ax.grid(alpha=0.3)
plt.show()

## Experimento 3 · Misma función, otros parámetros: otra molécula

Ahora la pregunta que nos lleva a MMFF94. Copia `U_molecula`, cambia **solo los parámetros** (en el código: `U_molecula_B` con enlace más blando y corto, $k = 200$, $r_0 = 0.95$ Å, ángulo $109.5°$ típico de carbono saturado) y observa:

- Las funciones son **idénticas** (resortes + Lennard-Jones).
- Los parámetros son distintos $\rightarrow$ la molécula es **químicamente diferente**.

Eso es precisamente el negocio de MMFF94: un **juego de parámetros por tipo de átomo y de enlace** (C–C, C–O, C=O, O–H, ...), obtenido de ajustar miles de datos experimentales. Cuando preparas un ligando y eliges "MMFF94", estás eligiendo ese juego de parámetros.

In [ ]:
# Misma función, otros parámetros = OTRA MOLÉCULA
def U_molecula_B(r1, r2, theta):
    # átomos más pequeños (sigma menor), acorde con su enlace más corto
    return (U_enlace(r1, 200.0, 0.95) + U_enlace(r2, 200.0, 0.95)
            + U_angulo(theta, 40.0, 109.5) + U_LJ(distancia_AC(r1, r2, theta), 0.3, 1.9))

r1s_B = np.linspace(0.7, 1.6, 400)
U_B = U_molecula_B(r1s_B, 0.95, 109.5)
i_B = np.argmin(U_B)
print(f"Molécula B: mínimo de enlace en r1 = {r1s_B[i_B]:.3f} Å (su r0 es 0.95 Å; el LJ lo empuja un poco, igual que en A)")

fig, ax = plt.subplots(figsize=(8, 4.2))
ax.plot(r1s, U_r1, color='darkred', lw=2, label='Molécula A: k=500, r0=1.1 Å (enlace rígido, largo)')
ax.plot(r1s_B, U_B, color='navy', lw=2, label='Molécula B: k=200, r0=0.95 Å (enlace blando, corto)')
ax.set_xlabel('r1 [Å]'); ax.set_ylabel('U [kcal/mol]')
ax.set_title('Mismas funciones, distintos parámetros → distintas moléculas')
ax.legend(); ax.grid(alpha=0.3)
plt.show()

print("\n💡 MMFF94 = estas mismas funciones con ~ parámetros por tipo de átomo/enlace, calibrados con experimentos.")

## Síntesis · El campo de fuerza pieza por pieza

| Término | Función | Significado | Parámetros (ej. MMFF94) |
|---|---|---|---|
| Enlace | $k(r-r_0)^2$ | resorte entre átomos unidos | $k$, $r_0$ por tipo de enlace |
| Ángulo | $k_\alpha(\theta-\theta_0)^2$ | resorte angular | $k_\alpha$, $\theta_0$ |
| Diedro | $V_n[1+\cos(n\phi-\gamma)]$ | rotación alrededor de enlaces | $V_n$, $n$, $\gamma$ |
| van der Waals | Lennard-Jones | atractivo + repulsión | $\varepsilon$, $\sigma$ por átomo |
| Electroestática | $q_i q_j / r_{ij}$ | cargas parciales | cargas $q_i$ |

> 🔑 La **forma de las funciones** es física básica; la **calidad de los parámetros** es lo que hace a un campo de fuerza útil (o no) para tu molécula. Ambas cosas importan por igual.

## Cierre · Conexión con el *docking molecular*

1. **El paso "minimizar MMFF94" del flujo de preparación de ligandos** ya no es una caja negra: es exactamente el Experimento 2 — relajar la geometría con un campo de fuerza hasta el mínimo más cercano.

2. **¿Por qué importa preparar bien el ligando?** Un ligando mal protonado, con malos ángulos o con tautomeros equivocados, se minimiza hacia un mínimo que no es el suyo → las poses del docking arrancan de una geometría errada. Basura entra, basura sale.

3. **¿Por qué MMFF94 y no otro campo?** Porque sus parámetros fueron calibrados precisamente sobre moléculas orgánicas pequeñas — el territorio de los ligandos. Para proteínas se usan AMBER, CHARMM, OPLS... (lo veremos en el mapa de campos de fuerza).

4. **¿Dónde estamos en el mapa del taller?**
   - Actividad 1: potencial → fuerza. ✅
   - Actividad 2: paisaje → minimización vs. optimización. ✅
   - **Actividad 3 (esta):** ensamblar la molécula completa → de Lennard-Jones a un campo de fuerza estilo MMFF94. ⬅️ *estás aquí*
   - Actividades 4–5: mover los átomos en el tiempo → dinámica molecular y auditoría de hipótesis de unión.

> 💬 **Pregunta guía del curso:** la honestidad de una hipótesis de unión empieza mucho antes del docking: empieza en la preparación. Ya sabes qué hace el campo de fuerza cuando "minimiza" tu ligando.